In [1]:
# Celda 1 — SIEMPRE primero, antes que nada
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import cv2, torch
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
torch.cuda.empty_cache()
print(f"VRAM libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")


VRAM libre: 15.8 GB


In [2]:
# Celda 2 — Datos
import pandas as pd
from utils.train import get_adversarial_dataloaders, train_model
from utils.models import Daowa_maadPrueba

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df_oxford   = pd.read_csv("labels/dataset_generated.csv")
df_ade20k   = pd.read_csv("labels/ade20k_adversarial_train.csv")
df_personas = pd.read_csv("labels/ade20k_person_negatives.csv")  # ← nuevo

df_ade20k   = df_ade20k.rename(columns={"image": "file"})
df_personas = df_personas.rename(columns={"image": "file"})

if 'is_gold' in df_oxford.columns:
    df_oxford = df_oxford[df_oxford['is_gold'] == False]

loaders = get_adversarial_dataloaders(
    df_oxford=df_oxford,
    df_ade20k=df_ade20k,
    df_personas=df_personas,          # ← nuevo
    oxford_dir="data/oxford",
    ade20k_dir="data/ADEChallengeData2016",
    batch_size=16,
    num_pos_per_batch=13,
    img_size=(256, 256),              # rápido para probar
    val_split=0.1,
    test_split=0.1,
)


ℹ️  Personas añadidas como hard negatives: 1515 train | 190 val | 189 test
✅ DataLoader adversarial listo — 5638 positivos | 9385 negativos | 433 batches/época (13P + 3N por batch)
✅ DataLoaders adversariales listos —
   Train : 5638 pos + 9385 neg
   Val   : 704 pos + 1174 neg
   Test  : 705 pos + 1172 neg


In [3]:
# Celda 3 — Modelo y entrenamiento
modelo = Daowa_maadPrueba(num_clases=1)
# Carga el mejor modelo adversarial de anoche
modelo.load_state_dict(torch.load(
    "Models/BestScore_Adv_2026-05-12_0.984.pth", map_location=device
), strict=False)

optimizer = torch.optim.AdamW(modelo.parameters(), lr=5e-6, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-7)

train_model(
    modelo=modelo,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    optimizador=optimizer,
    device=device,
    epochs=10,          # pocas para no sobrescribir lo aprendido
    patience=5,
    scheduler=scheduler,
    accum_steps=4,
    w_boundary=0.3,     # arranca más alto que antes (el modelo ya es estable)
)


C:\Users\PC\Desktop\Abbadon prueba SAM\env_python3_13\Lib\site-packages\torch\_utils.py:106: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  untyped_storage = torch.UntypedStorage(self.size(), device=device)


Output()

✓ Nuevo mejor IoU adversarial: 0.9999

✓ Nuevo mejor score global: 0.9834

Epoch 1: TrainLoss=0.0206 ValLoss=0.6339 | Acc_Pos=97.3% Acc_Neg=100.0% | IoU_Pos=0.9450 IoU_Neg=0.9999

...

Epoch 9: TrainLoss=-0.0748 ValLoss=0.3145 | Acc_Pos=97.3% Acc_Neg=99.9% | IoU_Pos=0.9454 IoU_Neg=1.0000

Pesos → Dice=0.56 Boundary=0.90

Epoch 10: TrainLoss=-0.0856 ValLoss=0.2749 | Acc_Pos=97.3% Acc_Neg=100.0% | IoU_Pos=0.9432 IoU_Neg=0.9997

Pesos → Dice=0.50 Boundary=1.00

'Entrenamiento adversarial v2 completado.'